# 03 — Idea 3: Dynamic TCO Under Climate Scenarios
## ML-Shifted Monte Carlo Distributions

**Research Question:** How does climate change (RCP 2.6, 4.5, 8.5) shift the
optimal datacenter location ranking when TCO distributions evolve dynamically
rather than remaining static?

**Approach:**
1. Generate climate projections for 10 locations under 3 RCP scenarios (2025-2050)
2. Train ML models to predict how climate variables shift TCO component distributions
3. Run dynamic Monte Carlo: year-by-year, the ML model adjusts power cost, PUE, and insurance
4. Compare static baseline vs climate-dynamic results

**Models:**
- Ensemble (XGBoost + Random Forest) — primary, interpretable
- Bayesian Neural Network (MC Dropout) — uncertainty quantification

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.location_profiles import load_locations
from src.data.climate_projections import generate_all_projections, load_scenarios
from src.tco.monte_carlo import run_all_locations
from src.tco.dynamic_distributions import run_dynamic_mc
from src.models.idea3.trainer import train_ensemble
from src.risk.metrics import compute_risk_metrics, risk_premium
from src.risk.scenario_comparator import build_comparison_table, rank_locations, climate_cost_premium_table

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

locations = load_locations()
SEED = 42

## 1. Climate Projections Under RCP Scenarios

In [ ]:
projections = generate_all_projections(seed=SEED)
print(f'Total projection rows: {len(projections)}')
print(f'Shape: {projections.shape}')
projections.head()

In [ ]:
# Temperature trajectories: Nordic vs Traditional under RCP 8.5
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

highlight = ['boden_sweden', 'atlanta_georgia', 'johor_malaysia', 'evanston_wyoming']

for ax, scenario in zip(axes, ['rcp26', 'rcp85']):
    s_df = projections[projections['scenario'] == scenario]
    for loc_key in highlight:
        loc_df = s_df[s_df['location_key'] == loc_key].sort_values('year')
        ax.plot(loc_df['year'], loc_df['avg_temp_c'], label=locations[loc_key].name, linewidth=2)
    ax.set_title(f'Temperature: {scenario.upper()}', fontsize=13)
    ax.set_xlabel('Year')
    ax.set_ylabel('Avg Temperature (°C)')
    ax.legend(fontsize=9)

fig.suptitle('Climate Projections: Why Location Choice Matters More Over Time', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# PUE degradation comparison
fig, ax = plt.subplots(figsize=(12, 6))
rcp85 = projections[projections['scenario'] == 'rcp85']

for loc_key in highlight:
    loc_df = rcp85[rcp85['location_key'] == loc_key].sort_values('year')
    ax.plot(loc_df['year'], loc_df['projected_pue'], label=locations[loc_key].name, linewidth=2)

ax.set_xlabel('Year')
ax.set_ylabel('Projected PUE')
ax.set_title('PUE Degradation Under RCP 8.5 (Business as Usual)', fontsize=14)
ax.legend()
plt.tight_layout()
plt.show()

## 2. Static Baseline Monte Carlo

In [ ]:
static_results = run_all_locations(locations, n_simulations=10000, seed=SEED)

print('Static MC Baseline (10-year TCO):')
for key in sorted(static_results, key=lambda k: static_results[k].mean):
    r = static_results[key]
    print(f'  {locations[key].name:30s}: mean={r.mean:8.1f}M  CV={r.cv:.4f}')

## 3. Train Ensemble Model

In [ ]:
model, metrics = train_ensemble(projections, seed=SEED)

print('Ensemble Model Validation R²:')
for target, score in metrics.get('val_r2', {}).items():
    print(f'  {target}: {score:.4f}')

# Feature importance
fi = model.feature_importance()
if fi:
    print('\nFeature Importance (per target):')
    for target, importances in fi.items():
        feat_names = metrics.get('feature_names', [f'f{i}' for i in range(len(importances))])
        ranked = sorted(zip(feat_names, importances), key=lambda x: -x[1])[:5]
        print(f'  {target}:')
        for name, imp in ranked:
            print(f'    {name}: {imp:.4f}')

## 4. Dynamic Monte Carlo Under Climate Scenarios

In [ ]:
# Run dynamic MC for all locations x scenarios
dynamic_results = {}  # {location: {scenario: distribution}}

for loc_key, loc in locations.items():
    dynamic_results[loc_key] = {}
    for scenario in ['rcp26', 'rcp45', 'rcp85']:
        loc_proj = projections[
            (projections['location_key'] == loc_key) &
            (projections['scenario'] == scenario)
        ]
        if len(loc_proj) == 0:
            continue
        result = run_dynamic_mc(
            location=loc, climate_projections=loc_proj,
            model=model, scenario=scenario,
            n_simulations=5000, seed=SEED,
        )
        dynamic_results[loc_key][scenario] = result.tco_distribution

print('Dynamic MC complete for all locations x scenarios')

In [ ]:
# Static vs Dynamic comparison for key locations
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
scenario_colors = {'rcp26': '#2ecc71', 'rcp45': '#f39c12', 'rcp85': '#e74c3c'}

for ax, loc_key in zip(axes.flat, ['boden_sweden', 'atlanta_georgia', 'evanston_wyoming', 'johor_malaysia']):
    static_dist = static_results[loc_key].tco_distribution
    ax.hist(static_dist, bins=50, alpha=0.4, color='gray', label=f'Static (μ={np.mean(static_dist):.0f}M)')
    
    for scenario, dist in dynamic_results.get(loc_key, {}).items():
        color = scenario_colors[scenario]
        ax.hist(dist, bins=50, alpha=0.35, color=color,
                label=f'{scenario.upper()} (μ={np.mean(dist):.0f}M)')
    
    ax.set_title(locations[loc_key].name, fontsize=13)
    ax.set_xlabel('10-Year TCO (Millions)')
    ax.legend(fontsize=8)

fig.suptitle('Static vs Climate-Dynamic TCO Distributions', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

## 5. Climate Cost Premium Analysis

In [ ]:
static_dists = {k: v.tco_distribution for k, v in static_results.items()}
premium_table = climate_cost_premium_table(static_dists, dynamic_results)
premium_table = premium_table.sort_values(['scenario', 'climate_premium_pct'], ascending=[True, False])
display(premium_table)

# Heatmap of climate premium %
pivot = premium_table.pivot(index='location', columns='scenario', values='climate_premium_pct')
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax, cbar_kws={'label': 'Climate Premium (%)'})
ax.set_title('Climate Cost Premium by Location and Scenario (%)', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Ranking Shift Analysis

In [ ]:
# Build full comparison table
all_dists = {}
for loc_key in locations:
    all_dists[loc_key] = {'static': static_results[loc_key].tco_distribution}
    all_dists[loc_key].update(dynamic_results.get(loc_key, {}))

comparison = build_comparison_table(all_dists)
ranked = rank_locations(comparison)

# Show ranking changes
print('\n--- Location Rankings by Scenario ---')
for scenario in ['static', 'rcp26', 'rcp45', 'rcp85']:
    s_ranked = ranked[ranked['scenario'] == scenario].sort_values('rank')
    print(f'\n{scenario.upper()}:')
    for _, row in s_ranked.iterrows():
        print(f"  #{int(row['rank']):2d}  {row['location']:30s}  TCO={row['mean_tco']:8.1f}M  CV={row['cv']:.4f}")

## 7. Key Findings

**Core insight:** Climate change doesn't just increase costs uniformly — it *widens the gap*
between climate-advantaged (Nordic) and climate-vulnerable (tropical/southern) locations.

| Finding | Impact |
|---------|--------|
| Nordic locations see minimal TCO increase under all scenarios | Their cold climate buffers PUE degradation |
| Atlanta, Johor face highest climate premiums under RCP 8.5 | Already-hot locations suffer compound effects |
| Rankings shift: US secondary markets become less competitive | Wyoming's advantage narrows as temperatures rise |
| RCP pathway matters enormously for investment decisions | Difference between RCP 2.6 and 8.5 is hundreds of millions |